# 11. MCP (Model Context Protocol)


## Learning Objectives

Learn how to connect external tools and context to an agent through MCP (Model Context Protocol).

This notebook covers:
- Understanding the MCP concept and architecture (server / client / host)
- Connecting to MCP servers with the `langchain-mcp-adapters` package
- Integrating MCP tools with an agent through `ChatOpenAI.bind_tools(mcp_tools)`
- Understanding the difference between stdio and SSE transports
- Connecting to multiple MCP servers at once


## 11.1 Environment Setup


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

print("환경 준비 완료.")

In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: automatically enabled when LANGSMITH_TRACING=true
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: pass config={"callbacks": [langfuse_handler]} to invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 11.2 MCP Concepts

**MCP (Model Context Protocol)** is an open protocol for providing external tools and context to an LLM in a **standardized way**.

### Architecture components

| Component | Role | Example |
|----------|------|------|
| **MCP server** | Exposes tools, resources, and prompts | File system server, DB server, API wrapper |
| **MCP client** | Connects to the server and fetches tools | `MultiServerMCPClient` |
| **Host** | Manages the client and connects it to the LLM | LangChain agent, IDE |

### Core resource types

- **Tools**: executable functions the agent can call
- **Resources**: data such as files or database records (converted to LangChain Blob objects)
- **Prompts**: reusable prompt templates

### Why MCP?

Before MCP, each tool needed its own custom integration code. MCP unifies this into **one standard protocol**, which means:
- Tool providers only need to implement an MCP server once
- LLM hosts can access every tool through one MCP client
- Tools can be reused across the ecosystem


## 11.3 Installing `langchain-mcp-adapters`

To use MCP from LangChain, you need the `langchain-mcp-adapters` package.


In [ ]:
# MCP 어댑터 설치 명령어
print("MCP 어댑터 설치:")
print("  uv add langchain-mcp-adapters mcp")
print()
print("주요 컴포넌트:")
print("  - MultiServerMCPClient: 여러 MCP 서버를 관리하는 클라이언트")
print("  - load_mcp_tools(session): MCP 세션을 LangChain Tool로 변환")
print("  - FastMCP: 빠르게 MCP 서버를 만드는 서버 유틸리티")

## 11.4 Stdio Transport

**Stdio (Standard I/O)** transport communicates with an MCP server through a local subprocess. It is a good fit for development and testing environments.


In [ ]:
from pathlib import Path; import json, tempfile, sys
server_path = Path(tempfile.gettempdir()) / "lc_mcp_math_server.py"
server_path.write_text('from mcp.server.fastmcp import FastMCP\nmcp = FastMCP("math")\n@mcp.tool()\ndef add(a: int, b: int) -> int:\n    return a + b\nif __name__ == "__main__":\n    mcp.run(transport="stdio")')
stdio_config = {"math": {"transport": "stdio", "command": sys.executable, "args": [str(server_path)]}}
print("Stdio 전송 설정:"); print(json.dumps(stdio_config, indent=2))
print(f"\n서버 파일: {server_path}")

## 11.5 SSE / HTTP Transport

**HTTP (streamable-http)** transport uses web-based communication and is a good fit for remote MCP servers. It also supports authentication headers and custom settings.


In [ ]:
# HTTP/streamable-http 전송 설정 예시
http_config = {
    "weather_server": {"transport": "streamable_http", "url": "https://weather-mcp.example.com/mcp", "headers": {"Authorization": "Bearer YOUR_API_KEY"}}
}
import json; print("HTTP 전송 설정:"); print(json.dumps(http_config, indent=2))
print("\n사용 패턴: client = MultiServerMCPClient(http_config) -> await client.get_tools()")

## 11.6 Loading MCP Tools and Integrating Them with an Agent

This is the common pattern for binding tools fetched from an MCP server into a LangChain agent.


In [ ]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

async def run_math_agent():
    client = MultiServerMCPClient(stdio_config)
    agent = create_agent(model=model, tools=await client.get_tools(), system_prompt="You can use MCP math tools.")
    return await agent.ainvoke(
        {"messages": [{"role": "user", "content": "Use the add tool to add 2 and 3."}]},
        config=lf_config,
    )

result = await run_math_agent()
print(result["messages"][-1].content)


## 11.7 Connecting to Multiple MCP Servers

As the name suggests, `MultiServerMCPClient` can manage several MCP servers at the same time.


In [ ]:
# 다중 MCP 서버 설정 예시
import json, sys
multi_server_config = {"math_server": {"transport": "stdio", "command": sys.executable, "args": [str(server_path)]}, "weather_server": {"transport": "streamable_http", "url": "https://weather-mcp.example.com/mcp"}, "database_server": {"transport": "stdio", "command": "npx", "args": ["-y", "@modelcontextprotocol/server-postgres"], "env": {"DATABASE_URL": "postgresql://..."}}}
print("다중 MCP 서버 설정:"); print(json.dumps(multi_server_config, indent=2, ensure_ascii=False))
print("\n사용 패턴: client = MultiServerMCPClient(multi_server_config) -> await client.get_tools()")
print("참고: 기본적으로 stateless — 각 도구 호출마다 새 세션 생성 후 정리")

## 11.8 Tool Interceptors

A **Tool Interceptor** is middleware that intercepts MCP tool calls. It can access runtime context, modify requests and responses, and implement retry logic.

### Tool interceptor use cases

| Use Case | Description |
|----------|------|
| Auth injection | Pass user-specific tokens at runtime |
| Request transformation | Rewrite tool call parameters |
| Response filtering | Remove sensitive information |
| Retry logic | Retry automatically after failures |
| Logging | Trace tool calls |


In [ ]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import ToolCallInterceptor

class LoggingInterceptor(ToolCallInterceptor):
    async def __call__(self, request, handler):
        print(f"Tool call: {request.name} @ {request.server_name}")
        return await handler(request)

async def run_with_interceptor():
    client = MultiServerMCPClient(stdio_config, tool_interceptors=[LoggingInterceptor()])
    agent = create_agent(model=model, tools=await client.get_tools(), system_prompt="Use the math tools.")
    return await agent.ainvoke(
        {"messages": [{"role": "user", "content": "Use the add tool to add 7 and 8."}]},
        config=lf_config,
    )

result = await run_with_interceptor()
print(result["messages"][-1].content)


## 11.9 Writing a Custom MCP Server

With the **FastMCP** library, you can build an MCP server quickly using decorators.


In [ ]:
from pathlib import Path
import tempfile
import sys
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

fastmcp_path = Path(tempfile.gettempdir()) / "lc_fastmcp_server.py"
fastmcp_path.write_text("""from mcp.server.fastmcp import FastMCP
mcp = FastMCP("my-tools")
@mcp.tool()
def add(a: int, b: int) -> int:
    return a + b
@mcp.tool()
def multiply(a: int, b: int) -> int:
    return a * b
@mcp.resource("config://app")
def get_config() -> str:
    return '{"version": "1.0", "debug": false}'
if __name__ == "__main__":
    mcp.run(transport="stdio")
""")

async def run_custom_server():
    client = MultiServerMCPClient({"my_tools": {"transport": "stdio", "command": sys.executable, "args": [str(fastmcp_path)]}})
    agent = create_agent(model=model, tools=await client.get_tools(), system_prompt="Use the multiplication tool.")
    return await agent.ainvoke(
        {"messages": [{"role": "user", "content": "Use the multiply tool to multiply 6 and 7."}]},
        config=lf_config,
    )

result = await run_custom_server()
print(result["messages"][-1].content)


## 11.10 Summary

This notebook covered:

| Topic | Key Idea |
|------|----------|
| **MCP concepts** | An open protocol that provides external tools and context to an LLM in a standardized way |
| **Stdio transport** | Local subprocess communication, good for development and testing |
| **SSE/HTTP transport** | Web-based communication for remote servers and authentication scenarios |
| **Agent integration** | Connect with `client.get_tools()` → `create_agent(tools=mcp_tools)` |
| **Multi-server support** | Use `MultiServerMCPClient` to manage several servers at once |
| **Interceptors** | Apply middleware for auth, logging, and request/response modification |
| **Custom servers** | Build an MCP server quickly with FastMCP decorators |

### Next Steps
→ Continue to **[12_frontend_streaming.ipynb](./12_frontend_streaming.ipynb)**


---
**References:**
- [MCP (Model Context Protocol)](../docs/langchain/16-mcp.md)
